In [0]:
bronze_dir = dbutils.widgets.get("p_folder_bronze")
silver_dir = dbutils.widgets.get("p_folder_silver")

'''storage_account_name = "storageaccountswec001"
bronze_dir = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/"'''


---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-6752565258245906>, line 1
----> 1 bronze_dir = dbutils.widgets.get("p_folder_bronze")
      2 silver_dir = dbutils.widgets.get("p_folder_silver")
      4 '''storage_account_name = "storageaccountswec001"
      5 bronze_dir = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/"'''

File /databricks/python_shell/lib/dbruntime/WidgetHandlerImpl.py:82, in WidgetsHandlerImpl.get(self, name)
     42 def get(self, name: str) -> str:
     43     """ Returns the current value of a widget with the given name.
     44 
     45     :param name: Name of the argument to be accessed
   (...)
     80         ```
     81     """
---> 82     return self._notebookArguments.getArgument(name, self._entry_point.getCurrentBindings())

File /databricks/spark/python/lib/py4j-0.10.

In [0]:
from pyspark.sql.functions import *

# Kolumna pośrednia: nazwa produktu po usunięciu koloru, z niej wyciągamy rozmiar
_product_no_color = when(
    col("Color").isNull() | upper(trim(col("Color"))).isin("", "NA", "N/A"),
    col("Product")
).otherwise(
    replace(col("Product"), col("Color"), lit(""))
)

_size_raw = regexp_extract(
    _product_no_color,
    r"(?i)[,-]\s*(\d+(?:\.\d+)?(?:\s*oz\.?)?|X{0,3}[SML]|Small|Medium|Large)\s*$",
    1
)

transformations = {
    "Product.csv": {
        "Standard Cost": regexp_replace(
            regexp_replace(col("Standard Cost"), r"\$", ""),
            ",",
            ""
        ).cast("double"),

        "Size": (
            when(trim(_size_raw) == "", "NA")
            .when(upper(trim(_size_raw)) == "EXTRA SMALL", "XS")
            .when(upper(trim(_size_raw)) == "SMALL", "S")
            .when(upper(trim(_size_raw)) == "MEDIUM", "M")
            .when(upper(trim(_size_raw)) == "LARGE", "L")
            .when(upper(trim(_size_raw)) == "EXTRA LARGE", "XL")
            .when(upper(trim(_size_raw)) == "EXTRA EXTRA LARGE", "XXL")
            .otherwise(regexp_replace(trim(_size_raw), r"\.$", ""))  # "30 oz." -> "30 oz"
            .cast("string")
        ),

        "Product": expr(r"""
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        CASE
                            WHEN Color IS NULL
                            OR UPPER(TRIM(Color)) IN ('', 'NA', 'N/A')
                            THEN Product
                            ELSE REPLACE(Product, Color, '')
                        END,
                        '(?i)[,-]\\s*(\\d+(?:\\.\\d+)?(?:\\s*oz\\.?)?|X{0,3}[SML]|Small|Medium|Large)\\s*$',
                        ''
                    ),
                    '[\\s,;:/|_-]+$',
                    ''
                )
            )
        """)
    },

    "Region.csv": {
        "CountryCode": expr("""
            CASE Country
                WHEN 'United States' THEN 'US'
                WHEN 'United Kingdom' THEN 'UK'
                ELSE UPPER(SUBSTRING(Country, 1, 3))
            END
        """)
    },

    "Reseller.csv": {
        "CountryCode": expr("""
            CASE `Country-Region`
                WHEN 'United States' THEN 'US'
                WHEN 'United Kingdom' THEN 'UK'
                ELSE UPPER(SUBSTRING(`Country-Region`, 1, 3))
            END
        """)
    },

    "Sales.csv": {
        "Unit Price": regexp_replace(
            regexp_replace(col("Unit Price"), r"\$", ""),
            ",",
            ""
        ).try_cast("double"),

        "Sales": regexp_replace(
            regexp_replace(col("Sales"), r"\$", ""),
            ",",
            ""
        ).cast("double"),

        "Cost": regexp_replace(
            regexp_replace(col("Cost"), r"\$", ""),
            ",",
            ""
        ).cast("double"),

        "OrderDate": try_to_date(
            regexp_replace(col("OrderDate"), r"^[A-Za-z]+,\s*", ""),
            "MMMM d, yyyy"
        )
    },

    "Salesperson.csv": {
        "EmployeeID": col("EmployeeID").cast("string")
    },

    "Targets.csv": {
        "EmployeeID": col("EmployeeID").cast("string"),

        "Target": regexp_replace(
            regexp_replace(col("Target"), r"\$", ""),
            ",",
            ""
        ).cast("double"),

        "TargetMonth": try_to_date(
            regexp_replace(col("TargetMonth"), r"^[A-Za-z]+,\s*", ""),
            "MMMM d, yyyy"
        )
    }
}

In [0]:
from pyspark.sql.functions import col, regexp_replace, expr

#dbutils.fs.rm(silver_dir, recurse=True)

for path in dbutils.fs.ls(bronze_dir):
    name = path.name
    path = path.path
    
    df = spark.read.format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .option("delimiter", "\t")\
        .option("encoding", "UTF-8")\
        .load(path)

    try:
        if name in transformations:
            df = df.withColumns(transformations.get(name))
        
        for column in df.columns:
            df = df.withColumnRenamed(column, column.replace(" ", "").replace("-", ""))
        
        df.write\
            .mode("overwrite") \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .save(silver_dir + name)

    except Exception as e:
        raise f"Error on loading {path}: {e}"


True
abfss://bronze@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/Product.csv


ProductKey,Product,StandardCost,Color,Subcategory,Category,BackgroundColorFormat,FontColorFormat
210,"HL Road Frame - Black, 58",868.63,Black,Road Frames,Components,#000000,#FFFFFF
215,"Sport-100 Helmet, Black",12.03,Black,Helmets,Accessories,#000000,#FFFFFF
216,"Sport-100 Helmet, Black",13.88,Black,Helmets,Accessories,#000000,#FFFFFF
217,"Sport-100 Helmet, Black",13.09,Black,Helmets,Accessories,#000000,#FFFFFF
253,"LL Road Frame - Black, 58",176.2,Black,Road Frames,Components,#000000,#FFFFFF
254,"LL Road Frame - Black, 58",170.14,Black,Road Frames,Components,#000000,#FFFFFF
255,"LL Road Frame - Black, 58",204.63,Black,Road Frames,Components,#000000,#FFFFFF
256,"LL Road Frame - Black, 60",176.2,Black,Road Frames,Components,#000000,#FFFFFF
257,"LL Road Frame - Black, 60",170.14,Black,Road Frames,Components,#000000,#FFFFFF
258,"LL Road Frame - Black, 60",204.63,Black,Road Frames,Components,#000000,#FFFFFF


True
abfss://bronze@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/Region.csv


SalesTerritoryKey,Region,Country,Group,CountryCode
1,Northwest,United States,North America,US
2,Northeast,United States,North America,US
3,Central,United States,North America,US
4,Southwest,United States,North America,US
5,Southeast,United States,North America,US
6,Canada,Canada,North America,CAN
7,France,France,Europe,FRA
8,Germany,Germany,Europe,GER
9,Australia,Australia,Pacific,AUS
10,United Kingdom,United Kingdom,Europe,UK


True
abfss://bronze@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/Reseller.csv


...

EmployeeKey,EmployeeID,Salesperson,Title,UPN
272,502097814,Stephen Jiang,North American Sales Manager,stephen-jiang@adventureworks.com
277,112432117,Brian Welcker,Director of Sales,brian-welcker@adventureworks.com
281,841560125,Michael Blythe,Sales Representative,michael-blythe@adventureworks.com
282,191644724,Linda Mitchell,Sales Representative,linda-mitchell@adventureworks.com
283,615389812,Jillian Carson,Sales Representative,jillian-carson@adventureworks.com
284,234474252,Garrett Vargas,Sales Representative,garrett-vargas@adventureworks.com
285,716374314,Tsvi Reiter,Sales Representative,tsvi-reiter@adventureworks.com
286,61161660,Pamela Ansman-Wolfe,Sales Representative,pamela-ansman-wolfe@adventureworks.com
287,139397894,Shu Ito,Sales Representative,shu-ito@adventureworks.com
288,399771412,José Saraiva,Sales Representative,jose-saraiva@adventureworks.com


False
abfss://bronze@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/SalespersonRegion.csv


EmployeeKey,SalesTerritoryKey
272,1
272,2
272,3
272,4
272,5
272,6
277,1
277,2
277,3
277,4


True
abfss://bronze@storageaccountswec001.dfs.core.windows.net/adventureworkskaggle/ba1c6c7a-f29d-4427-9b37-7d449cad2153/Targets.csv


EmployeeID,Target,TargetMonth
90836195,500000.0,2017-12-01
112432117,500000.0,2017-07-01
139397894,500000.0,2017-12-01
191644724,500000.0,2017-09-01
502097814,500000.0,2017-07-01
716374314,500000.0,2017-12-01
841560125,500000.0,2017-08-01
987554265,500000.0,2017-12-01
61161660,500000.0,2018-02-01
90836195,500000.0,2018-05-01
